In [14]:
import os 
import pandas as pd 
import requests
from pathlib import Path
import json

# USA indicators

In [15]:
# Define the project directory.
ROOT = Path(
    r"C:\Christian_meus_estudos\financial_market_quantitative_analyses\pipeline_market"
)

# Load the API key from the configuration file.
config_path = ROOT / "config.json"

with open(config_path, encoding="utf-8") as file:
    config = json.load(file)

API_KEY = config["fred_api_key"]

# Define the economic series to retrieve.
series = {
    "CPIAUCSL": "cpi_index",
    "FEDFUNDS": "fed_funds_rate",
    "GDPC1": "real_gdp"
}

# Define the FRED API endpoint.
url = "https://api.stlouisfed.org/fred/series/observations"

dataframes = []

for series_id, indicator in series.items():

    # Set the API request parameters.
    params = {
        "series_id": series_id,
        "api_key": API_KEY,
        "file_type": "json",
        "observation_start": "2010-01-01"
    }

    # Send the request to the FRED API.
    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()

    # Extract the observations from the JSON response.
    observations = response.json()["observations"]

    # Convert the observations into a DataFrame.
    df = pd.DataFrame(observations)

    # Select and rename the required columns.
    df = df[["date", "value"]].rename(
        columns={"date": "reference_date"}
    )

    # Convert dates and values to the appropriate data types.
    df["reference_date"] = pd.to_datetime(df["reference_date"])
    df["value"] = pd.to_numeric(df["value"], errors="coerce")

    # Add metadata to identify each observation.
    df["country"] = "USA"
    df["series_id"] = series_id
    df["indicator"] = indicator

    # Store the DataFrame for consolidation.
    dataframes.append(df)

    print(f"Retrieved {len(df)} observations for {series_id}.")

# Combine all economic series into a single DataFrame.
df_macro = pd.concat(dataframes, ignore_index=True)

# Pivot the data so each economic indicator becomes a column.
df_macro = df_macro.pivot(
    index="reference_date",
    columns="indicator",
    values="value"
).reset_index()

# Remove the columns index name.
df_macro.columns.name = None

# Define the output directory and CSV file.
storage_dir = ROOT / "storage"
storage_dir.mkdir(parents=True, exist_ok=True)

output = storage_dir / "usa_macro.csv"

# Save the consolidated data to CSV.
df_macro.to_csv(output, index=False)

# Display a sample of the resulting DataFrame.
print(df_macro.head(10))

print(f"\nFile saved to: {output}")

Retrieved 201 observations for CPIAUCSL.
Retrieved 201 observations for FEDFUNDS.
Retrieved 66 observations for GDPC1.
  reference_date  cpi_index  fed_funds_rate   real_gdp
0     2010-01-01    217.488            0.11  16582.710
1     2010-02-01    217.281            0.13        NaN
2     2010-03-01    217.353            0.16        NaN
3     2010-04-01    217.403            0.20  16743.162
4     2010-05-01    217.290            0.20        NaN
5     2010-06-01    217.199            0.18        NaN
6     2010-07-01    217.605            0.18  16872.266
7     2010-08-01    217.923            0.19        NaN
8     2010-09-01    218.275            0.19        NaN
9     2010-10-01    219.035            0.19  16960.864

File saved to: C:\Christian_meus_estudos\financial_market_quantitative_analyses\pipeline_market\storage\usa_macro.csv


# BCE indicators

In [16]:
# Define the project directory.
ROOT = Path(
    r"C:\Christian_meus_estudos\financial_market_quantitative_analyses\pipeline_market"
)

# Define the output directory.
STORAGE_DIR = ROOT / "storage"
STORAGE_DIR.mkdir(parents=True, exist_ok=True)

# Define the ECB API endpoint.
BASE_URL = "https://data-api.ecb.europa.eu/service/data"

# Define the economic indicators to retrieve.
series = {
    "inflation_yoy": {
        "dataset": "HICP",
        "key": "M.U2.N.000000.4D0.ANR",
        "start_period": "2010-01",
    },
    "ecb_deposit_rate": {
        "dataset": "FM",
        "key": "D.U2.EUR.4F.KR.DFR.LEV",
        "start_period": "2010-01-01",
    },
    "real_gdp_growth_qoq": {
        "dataset": "MNA",
        "key": "Q.Y.I10.W2.S1.S1.B.B1GQ._Z._Z._Z.EUR.LR.G1",
        "start_period": "2010-Q1",
    },
}

dataframes = []

# Retrieve each economic indicator.
for indicator, settings in series.items():

    # Build the API request URL.
    url = (
        f"{BASE_URL}/"
        f"{settings['dataset']}/"
        f"{settings['key']}"
    )

    # Set the request parameters.
    params = {
        "startPeriod": settings["start_period"],
        "format": "csvdata",
    }

    # Send the request to the ECB API.
    response = requests.get(
        url,
        params=params,
        timeout=60,
    )
    response.raise_for_status()

    # Read the CSV response into a DataFrame.
    df = pd.read_csv(io.StringIO(response.text))

    # Keep the observation date and value.
    df = df[["TIME_PERIOD", "OBS_VALUE"]].rename(
        columns={
            "TIME_PERIOD": "reference_date",
            "OBS_VALUE": indicator,
        }
    )

    # Convert the columns to appropriate data types.
    df["reference_date"] = pd.to_datetime(
        df["reference_date"],
        errors="coerce",
    )
    df[indicator] = pd.to_numeric(
        df[indicator],
        errors="coerce",
    )

    # Remove observations without valid dates.
    df = df.dropna(subset=["reference_date"])

    # Store the indicator DataFrame.
    dataframes.append(df)

    print(
        f"Retrieved {len(df)} observations "
        f"for {indicator}."
    )

# Merge all indicators using their observation dates.
df_macro = dataframes[0]

for df in dataframes[1:]:
    df_macro = df_macro.merge(
        df,
        on="reference_date",
        how="outer",
    )

# Sort the observations chronologically.
df_macro = df_macro.sort_values(
    "reference_date"
).reset_index(drop=True)

# Save the consolidated dataset.
output = STORAGE_DIR / "euro_macro.csv"

df_macro.to_csv(output, index=False)

# Display the resulting dataset.
print("\nSample of retrieved data:")
print(df_macro.head(10))

print("\nDataset information:")
print(df_macro.info())

print(f"\nFile saved to: {output}")


Retrieved 201 observations for inflation_yoy.
Retrieved 6135 observations for ecb_deposit_rate.
Retrieved 66 observations for real_gdp_growth_qoq.

Sample of retrieved data:
  reference_date  inflation_yoy  ecb_deposit_rate  real_gdp_growth_qoq
0     2010-01-01            1.0              0.25             0.408366
1     2010-01-02            NaN              0.25                  NaN
2     2010-01-03            NaN              0.25                  NaN
3     2010-01-04            NaN              0.25                  NaN
4     2010-01-05            NaN              0.25                  NaN
5     2010-01-06            NaN              0.25                  NaN
6     2010-01-07            NaN              0.25                  NaN
7     2010-01-08            NaN              0.25                  NaN
8     2010-01-09            NaN              0.25                  NaN
9     2010-01-10            NaN              0.25                  NaN

Dataset information:
<class 'pandas.core.fra

C:\Users\Azog\AppData\Local\Temp\ipykernel_13160\2321326225.py:70: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["reference_date"] = pd.to_datetime(


# Switzerland indicators

In [17]:
# Define the project directory.
ROOT = Path(
    r"C:\Christian_meus_estudos\financial_market_quantitative_analyses\pipeline_market"
)

# Define the storage directory.
STORAGE_DIR = ROOT / "storage"
STORAGE_DIR.mkdir(parents=True, exist_ok=True)

# Define the SNB API base URL.
BASE_URL = "https://data.snb.ch/api/cube"

# Define the economic indicators to retrieve.
series = {
    "inflation_yoy": {
        "cube": "plkopr",
        "params": {"dimSel": "D0(VVP)", "fromDate": "2010-01"},
    },
    "snb_policy_rate": {
        "cube": "snboffzisa",
        "params": {"dimSel": "D0(LZ)", "fromDate": "2010-01"},
    },
    "real_gdp": {
        "cube": "gdprpq",
        "params": {"fromDate": "2010-Q1"},
    },
}



def download_series(cube, params):
    # Build the API URL.
    url = f"{BASE_URL}/{cube}/data/csv/en"

    # Send the API request.
    response = requests.get(url, params=params, timeout=60)
    response.raise_for_status()

    # Check the response before parsing it.
    if not response.text.strip():
        raise ValueError(f"The API returned an empty response for {cube}.")

    # Read the semicolon-delimited CSV after its metadata lines.
    df = pd.read_csv(
        StringIO(response.text.lstrip("\ufeff")),
        sep=";",
        skiprows=2,
    )

    # Normalize column names.
    df.columns = [
        str(column).strip().lower().replace(" ", "_")
        for column in df.columns
    ]

    # Identify the date column.
    date_column = next(
        (
            column
            for column in df.columns
            if column in {"date", "time_period", "period", "reference_date"}
        ),
        None,
    )

    # Identify the observation value column.
    value_column = next(
        (
            column
            for column in df.columns
            if column in {"value", "obs_value", "observation_value"}
        ),
        None,
    )

    # Display diagnostic information if the format is unexpected.
    if date_column is None or value_column is None:
        raise ValueError(
            f"Unexpected CSV structure for {cube}.\n"
            f"Columns: {df.columns.tolist()}\n"
            f"Response preview:\n{response.text[:1000]}"
        )

    # Keep and rename the required columns.
    df = df[[date_column, value_column]].rename(
        columns={
            date_column: "reference_date",
            value_column: "value",
        }
    )

    # Convert dates and values to appropriate data types.
    df["reference_date"] = pd.to_datetime(
        df["reference_date"], errors="coerce"
    )
    df["value"] = pd.to_numeric(df["value"], errors="coerce")

    # Remove invalid observations.
    df = df.dropna(subset=["reference_date", "value"])

    return df

dataframes = []

# Download each economic indicator.
for indicator, settings in series.items():
    print(f"Downloading {indicator}...")

    df = download_series(
        settings["cube"],
        settings["params"],
    )

    # Rename the value column to the indicator name.
    df = df.rename(columns={"value": indicator})

    # Store the downloaded indicator.
    dataframes.append(df)

    print(f"Retrieved {len(df)} observations for {indicator}.")

# Merge the indicators by reference date.
df_macro = dataframes[0]

for df in dataframes[1:]:
    df_macro = df_macro.merge(
        df,
        on="reference_date",
        how="outer",
    )

# Sort observations chronologically.
df_macro = df_macro.sort_values(
    "reference_date"
).reset_index(drop=True)

# Save the consolidated dataset.
output = STORAGE_DIR / "switzerland_macro.csv"
df_macro.to_csv(output, index=False)

# Display the resulting dataset.
print("\nSample of retrieved data:")
print(df_macro.head(10))

print("\nObservations per indicator:")
print(df_macro.count())

print(f"\nFile saved to: {output}")


Retrieved 200 observations for inflation_yoy.
Retrieved 87 observations for snb_policy_rate.
Retrieved 1980 observations for real_gdp.

Sample of retrieved data:
  reference_date  inflation_yoy  snb_policy_rate      real_gdp
0     2010-01-01       1.023634              NaN  80922.546920
1     2010-01-01       1.023634              NaN      1.039175
2     2010-01-01       1.023634              NaN      3.876533
3     2010-01-01       1.023634              NaN      0.317084
4     2010-01-01       1.023634              NaN      6.700540
5     2010-01-01       1.023634              NaN      5.403313
6     2010-01-01       1.023634              NaN      6.146819
7     2010-01-01       1.023634              NaN      0.744158
8     2010-01-01       1.023634              NaN      8.988417
9     2010-01-01       1.023634              NaN      1.761502

Observations per indicator:
reference_date     2114
inflation_yoy      2114
snb_policy_rate     899
real_gdp           1980
dtype: int64

File s

C:\Users\Azog\AppData\Local\Temp\ipykernel_13160\2683080849.py:93: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["reference_date"] = pd.to_datetime(


# Japan indicators

In [18]:
# Define the project directory.
ROOT = Path(
    r"C:\Christian_meus_estudos\financial_market_quantitative_analyses\pipeline_market"
)

# Load the FRED API key.
with open(ROOT / "config.json", encoding="utf-8") as file:
    config = json.load(file)

API_KEY = config["fred_api_key"]

# Define the FRED API endpoint.
URL = "https://api.stlouisfed.org/fred/series/observations"

# Define the economic series to retrieve.
# CPI and interest-rate series must be checked for their latest dates.
series = {
    "JPNCPIALLMINMEI": "cpi_index",
    "IRSTCI01JPM156N": "central_bank_rate",
    "JPNRGDPEXP": "real_gdp",
}

dataframes = []

# Retrieve each economic series.
for series_id, indicator in series.items():

    # Set the API request parameters.
    params = {
        "series_id": series_id,
        "api_key": API_KEY,
        "file_type": "json",
        "observation_start": "2010-01-01",
    }

    # Send the request to FRED.
    response = requests.get(URL, params=params, timeout=30)
    response.raise_for_status()

    # Parse the observations.
    observations = response.json()["observations"]

    # Convert the observations into a DataFrame.
    df = pd.DataFrame(observations)

    # Keep and rename the required columns.
    df = df[["date", "value"]].rename(
        columns={
            "date": "reference_date",
            "value": indicator,
        }
    )

    # Convert dates and values.
    df["reference_date"] = pd.to_datetime(
        df["reference_date"], errors="coerce"
    )
    df[indicator] = pd.to_numeric(
        df[indicator], errors="coerce"
    )

    # Remove missing observations.
    df = df.dropna(subset=["reference_date", indicator])

    # Display the available date range.
    if not df.empty:
        print(
            f"{indicator}: {df['reference_date'].min().date()} "
            f"to {df['reference_date'].max().date()}"
        )

    # Store the indicator DataFrame.
    dataframes.append(df)

# Merge the indicators by reference date.
df_macro = dataframes[0]

for df in dataframes[1:]:
    df_macro = df_macro.merge(
        df,
        on="reference_date",
        how="outer",
    )

# Calculate year-over-year CPI inflation.
df_macro = df_macro.sort_values("reference_date")

df_macro["inflation_yoy"] = (
    df_macro["cpi_index"]
    .pct_change(periods=12, fill_method=None)
    .mul(100)
)

# Remove the CPI index if only inflation is needed.
df_macro = df_macro.drop(columns=["cpi_index"])

# Reset the index.
df_macro = df_macro.reset_index(drop=True)

# Create the storage directory.
STORAGE_DIR = ROOT / "storage"
STORAGE_DIR.mkdir(parents=True, exist_ok=True)

# Save the consolidated dataset.
output = STORAGE_DIR / "japan_macro.csv"
df_macro.to_csv(output, index=False)

# Display the resulting data.
print("\nSample:")
print(df_macro.head(10))

print("\nAvailable observations:")
print(df_macro.count())

print(f"\nFile saved to: {output}")


cpi_index: 2010-01-01 to 2021-06-01
central_bank_rate: 2010-01-01 to 2026-09-01
real_gdp: 2010-01-01 to 2026-04-01

Sample:
  reference_date  central_bank_rate  real_gdp  inflation_yoy
0     2010-01-01              0.096  520844.0            NaN
1     2010-02-01              0.101       NaN            NaN
2     2010-03-01              0.097       NaN            NaN
3     2010-04-01              0.093  528053.7            NaN
4     2010-05-01              0.091       NaN            NaN
5     2010-06-01              0.095       NaN            NaN
6     2010-07-01              0.094  537192.1            NaN
7     2010-08-01              0.095       NaN            NaN
8     2010-09-01              0.091       NaN            NaN
9     2010-10-01              0.091  532541.3            NaN

Available observations:
reference_date       201
central_bank_rate    201
real_gdp              66
inflation_yoy        126
dtype: int64

File saved to: C:\Christian_meus_estudos\financial_market_quantita

In [19]:
# Define the project directory.
ROOT = Path(
    r"C:\Christian_meus_estudos\financial_market_quantitative_analyses\pipeline_market"
)

# Load the FRED API key.
with open(ROOT / "config.json", encoding="utf-8") as file:
    config = json.load(file)

API_KEY = config["fred_api_key"]

# Define the FRED API endpoint.
URL = "https://api.stlouisfed.org/fred/series/observations"

# Define the economic series.
series = {
    "CPALTT01CAM657N": "cpi_index",
    "IRSTCI01CAM156N": "central_bank_rate",
    "NGDPRSAXDCCAQ": "real_gdp",
}

dataframes = []

# Download each economic series.
for series_id, indicator in series.items():

    # Configure the API request.
    params = {
        "series_id": series_id,
        "api_key": API_KEY,
        "file_type": "json",
        "observation_start": "2010-01-01",
    }

    # Request observations from FRED.
    response = requests.get(URL, params=params, timeout=30)
    response.raise_for_status()

    # Parse the API response.
    observations = response.json()["observations"]

    # Create the DataFrame.
    df = pd.DataFrame(observations)

    # Select and rename the required columns.
    df = df[["date", "value"]].rename(
        columns={
            "date": "reference_date",
            "value": indicator,
        }
    )

    # Convert dates and values.
    df["reference_date"] = pd.to_datetime(
        df["reference_date"], errors="coerce"
    )
    df[indicator] = pd.to_numeric(
        df[indicator], errors="coerce"
    )

    # Remove invalid observations.
    df = df.dropna(
        subset=["reference_date", indicator]
    )

    # Report the retrieved date range.
    if not df.empty:
        print(
            f"{indicator}: "
            f"{df['reference_date'].min().date()} to "
            f"{df['reference_date'].max().date()}"
        )

    # Store the downloaded series.
    dataframes.append(df)

# Merge all series by reference date.
df_macro = dataframes[0]

for df in dataframes[1:]:
    df_macro = df_macro.merge(
        df,
        on="reference_date",
        how="outer",
    )

# Sort observations chronologically.
df_macro = df_macro.sort_values("reference_date")

# Calculate year-over-year CPI inflation.
df_macro["inflation_yoy"] = (
    df_macro["cpi_index"]
    .pct_change(periods=12, fill_method=None)
    * 100
)

# Calculate quarter-over-quarter real GDP growth.
# GDP observations are quarterly, so use quarterly observations.
gdp = (
    df_macro[["reference_date", "real_gdp"]]
    .dropna()
    .sort_values("reference_date")
)

gdp["real_gdp_growth_qoq"] = (
    gdp["real_gdp"]
    .pct_change(fill_method=None)
    * 100
)

# Merge the calculated GDP growth back into the dataset.
df_macro = df_macro.merge(
    gdp[["reference_date", "real_gdp_growth_qoq"]],
    on="reference_date",
    how="left",
)

# Keep the final indicators.
df_macro = df_macro[
    [
        "reference_date",
        "inflation_yoy",
        "central_bank_rate",
        "real_gdp_growth_qoq",
    ]
]

# Reset the index.
df_macro = df_macro.reset_index(drop=True)

# Create the storage directory.
STORAGE_DIR = ROOT / "storage"
STORAGE_DIR.mkdir(parents=True, exist_ok=True)

# Save the consolidated dataset.
output = STORAGE_DIR / "canada_macro.csv"
df_macro.to_csv(output, index=False)

# Display the results.
print("\nSample:")
print(df_macro.head(10))

print("\nAvailable observations:")
print(df_macro.count())

print(f"\nFile saved to: {output}")


cpi_index: 2010-01-01 to 2024-02-01
central_bank_rate: 2010-01-01 to 2026-09-01
real_gdp: 2010-01-01 to 2026-04-01

Sample:
  reference_date  inflation_yoy  central_bank_rate  real_gdp_growth_qoq
0     2010-01-01            NaN           0.244385                  NaN
1     2010-02-01            NaN           0.244295                  NaN
2     2010-03-01            NaN           0.244539                  NaN
3     2010-04-01            NaN           0.247281             0.523654
4     2010-05-01            NaN           0.245700                  NaN
5     2010-06-01            NaN           0.491891                  NaN
6     2010-07-01            NaN           0.600386             0.700032
7     2010-08-01            NaN           0.748733                  NaN
8     2010-09-01            NaN           0.951171                  NaN
9     2010-10-01            NaN           0.998505             1.121802

Available observations:
reference_date         201
inflation_yoy          156
centr

# England indicators

In [20]:
# Define the project directory.
ROOT = Path(
    r"C:\Christian_meus_estudos\financial_market_quantitative_analyses\pipeline_market"
)

# Load the FRED API key.
with open(ROOT / "config.json", encoding="utf-8") as file:
    config = json.load(file)

API_KEY = config["fred_api_key"]

# Define the FRED API endpoint.
URL = "https://api.stlouisfed.org/fred/series/observations"

# Define the economic series.
series = {
    "GBRCPIALLMINMEI": "cpi_index",
    "IRSTCI01GBM156N": "central_bank_rate",
    "NGDPRSAXDCGBQ": "real_gdp",
}

dataframes = []

# Retrieve each economic series.
for series_id, indicator in series.items():

    # Configure the API request.
    params = {
        "series_id": series_id,
        "api_key": API_KEY,
        "file_type": "json",
        "observation_start": "2010-01-01",
    }

    # Request observations from FRED.
    response = requests.get(URL, params=params, timeout=30)
    response.raise_for_status()

    # Parse the observations.
    observations = response.json()["observations"]
    df = pd.DataFrame(observations)

    # Keep and rename the required columns.
    df = df[["date", "value"]].rename(
        columns={
            "date": "reference_date",
            "value": indicator,
        }
    )

    # Convert dates and values.
    df["reference_date"] = pd.to_datetime(
        df["reference_date"], errors="coerce"
    )
    df[indicator] = pd.to_numeric(
        df[indicator], errors="coerce"
    )

    # Remove invalid observations.
    df = df.dropna(subset=["reference_date", indicator])

    # Report the available date range.
    if not df.empty:
        print(
            f"{indicator}: "
            f"{df['reference_date'].min().date()} to "
            f"{df['reference_date'].max().date()}"
        )

    # Store the downloaded series.
    dataframes.append(df)

# Merge the indicators by reference date.
df_macro = dataframes[0]

for df in dataframes[1:]:
    df_macro = df_macro.merge(
        df,
        on="reference_date",
        how="outer",
    )

# Sort observations chronologically.
df_macro = df_macro.sort_values("reference_date")

# Calculate year-over-year CPI inflation.
df_macro["inflation_yoy"] = (
    df_macro["cpi_index"]
    .pct_change(periods=12, fill_method=None)
    .mul(100)
)

# Calculate quarter-over-quarter real GDP growth.
gdp = (
    df_macro[["reference_date", "real_gdp"]]
    .dropna()
    .sort_values("reference_date")
    .copy()
)

gdp["real_gdp_growth_qoq"] = (
    gdp["real_gdp"]
    .pct_change(fill_method=None)
    .mul(100)
)

# Merge GDP growth into the main dataset.
df_macro = df_macro.merge(
    gdp[["reference_date", "real_gdp_growth_qoq"]],
    on="reference_date",
    how="left",
)

# Keep the final indicators.
df_macro = df_macro[
    [
        "reference_date",
        "inflation_yoy",
        "central_bank_rate",
        "real_gdp_growth_qoq",
    ]
]

# Create the storage directory.
STORAGE_DIR = ROOT / "storage"
STORAGE_DIR.mkdir(parents=True, exist_ok=True)

# Save the dataset.
output = STORAGE_DIR / "uk_macro.csv"
df_macro.to_csv(output, index=False)

# Display the results.
print("\nSample:")
print(df_macro.head(10))

print("\nAvailable observations:")
print(df_macro.count())

print(f"\nFile saved to: {output}")


cpi_index: 2010-01-01 to 2025-03-01
central_bank_rate: 2010-01-01 to 2026-09-01
real_gdp: 2010-01-01 to 2026-04-01

Sample:
  reference_date  inflation_yoy  central_bank_rate  real_gdp_growth_qoq
0     2010-01-01            NaN             0.4545                  NaN
1     2010-02-01            NaN             0.4672                  NaN
2     2010-03-01            NaN             0.4713                  NaN
3     2010-04-01            NaN             0.4726             1.142262
4     2010-05-01            NaN             0.4842                  NaN
5     2010-06-01            NaN             0.4870                  NaN
6     2010-07-01            NaN             0.4900             0.587887
7     2010-08-01            NaN             0.4910                  NaN
8     2010-09-01            NaN             0.5010                  NaN
9     2010-10-01            NaN             0.5083             0.050560

Available observations:
reference_date         201
inflation_yoy          171
centr